# List 3: Attention, Residuals & Normalization

Labs: Mon 9 Nov, Tue 17 Nov, Mon 23 Nov 2026 · Due: Sun 29 Nov 2026, 23:59 · 1 point

- For each task, show the requested numbers or plots and explain them in two to four sentences. The list is graded as a whole, in steps of 0.25, by how complete, correct and well explained your solution is. In Lab 7 some of you will be asked to explain one of your plots in two minutes.
- The whole list is one experiment. You start from the bigram model of List 1, as a character-level language model on `tinyshakespeare.txt`, and add one piece of the Transformer at a time: averaging the past, attention, queries/keys/values, positions, the causal mask, heads, depth, residual connections, LayerNorm, GELU. After every step you train for the same small budget and read the validation loss. The question is always the same: did the loss move, and why? Keep a table of your runs (model, parameters, train loss, validation loss, seconds); Task 7.4 asks for it.
- Data loading, the training loop and text generation are given in Materials 0.3 (about 50 lines). Use them, or your own `train` from List 1. Model code, structure and names are up to you. Defaults (context 64, batch 32, $d = 64$, 1000 steps of AdamW with learning rate $3 \cdot 10^{-3}$) are for when you have no preference; every run takes from seconds to a few minutes on a laptop CPU. Numbers given as "our run" are what we got with the defaults, so you know what to expect; yours will differ a little.
- Keep the signatures and attribute names of the four definitions `MultiHeadAttention`, `LayerNorm`, `GELU` and `FeedForward` (Materials, Appendix A), because List 4 builds the GPT model from them and List 5 loads GPT-2's weights into them.
- Background and the PyTorch you need for Task X.Y are in Materials X.Y.
- Submit this notebook run from top to bottom with outputs, or Python scripts with a PDF report of at most four pages (results and explanations per task). Late: −10% per day, 0 after 5 days.
- Tasks marked *(optional)* are not graded: the full point is available from the other tasks. In exceptional cases the grader may take them into account, within the 1-point maximum.
- Individual work. Name the AI tools you used at the top of your solution and attach the prompts you gave them at the end.

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn

## Lab 5: What attention computes

### 5.1 A bigram model has no past

Default setup for the whole list: characters of `tinyshakespeare.txt` as tokens ($V = 65$), the first 90% for training, windows of 64 characters, batches of 32, 1000 steps of AdamW with learning rate $3 \cdot 10^{-3}$ (Materials 0.3). Report train and validation loss in nats per character.

- Train the bigram model with the harness: a token embedding of size $d = 64$ followed by the output layer (Materials 0.3). Compare its validation loss with $\ln V$ and with the entropy of the empirical bigram distribution of the training text, $H(\text{next} \mid \text{current})$, computed from counts. Our run: 2.50; the bigram entropy is 2.45.
- Give the model the past in the simplest way: replace the embedding $x_t$ of every position by the mean of the embeddings at positions $0, \dots, t$, written as $WX$ with a lower-triangular matrix $W$ whose rows sum to 1. Train it. Our run: 3.17, worse than the bigram. Explain why: what does the average lose, and what happens to the current character as $t$ grows?

### 5.2 Let the token choose: attention without weights

- Replace the uniform weights by weights the data chooses: $a_{ts} = x_t^\top x_s$ for $s \le t$, $-\infty$ for $s > t$, $\alpha_{ts} = \operatorname{softmax}_s(a_{ts})$, $z_t = \sum_s \alpha_{ts} x_s$. Train the same model with this layer instead of the average. Our run: 2.50, the bigram's loss again. Look at the trained weights: how much weight does a character put on itself, and why is that the best this layer can do?
- Pick a window, shuffle the characters at positions $0, \dots, t-1$ and keep position $t$: show that the logits of your trained model at position $t$ do not change. Then prove in two lines that $f(PX) = Pf(X)$ for unmasked attention $f(X) = \operatorname{softmax}(XX^\top)X$ and a permutation matrix $P$. What does this mean for `dog bites man` and `man bites dog`?

### 5.3 Queries, keys, values, positions, and the scale

- Give every token three roles: $q_t = W_Q x_t$, $k_s = W_K x_s$, $v_s = W_V x_s$, scores $q_t^\top k_s / \sqrt{d_k}$, with $W_Q, W_K, W_V$ as `nn.Linear(d, d, bias=False)` (one head, $d_k = d_v = d = 64$). Add a learned positional embedding `nn.Embedding(context, d)` as in List 2, Task 4.3. Train both versions, with and without positions. Our run: 2.52 without, 2.40 with. Explain what the three projections can express that weight-free attention cannot (think of "look at the previous character"), and why positions are needed for that.
- Look at what the head learned: for one window of the validation text, plot the $64 \times 64$ attention matrix of the trained head as an image with the characters as tick labels (Materials 5.3). Describe the pattern in two sentences: which positions does a character look at?
- Train the model with positions once more without the division by $\sqrt{d_k}$. Our run: 2.66 instead of 2.40. Explain the loss with the softmax: what happens to the attention weights and to their gradient $\partial \alpha / \partial a$ when the scores are large, and why does this get worse as $d_k$ grows (Materials 5.3)?

- **Checkpoint:** your one-head model with positions (Task 5.3) has a validation loss below your bigram model after the same 1000 steps; every row of its attention matrix sums to 1; and shuffling the past leaves the logits of the model without positions unchanged (Materials, Appendix B).

## Lab 6: Causality and heads

### 6.1 Break the mask

- Remove the causal mask from your one-head model and train it. Report train and validation loss and generate 200 characters. Our run: 0.05 on both splits. Explain exactly where the model finds the answer: which input does position $t$ see, and what is its target?
- Write `leak_test(model)`: feed 20 batches of uniformly random tokens and return the mean cross-entropy. A model that cannot see the future cannot get below $\ln V = 4.17$ (an untrained one scores about $\ln V$, a trained one above it, because it expects Shakespeare); a leaking model gets much less. Run it on both models. Keep the function; it is part of the checkpoints of Labs 6 and 7.
- Compute the causal weights in two ways: (a) scores above the diagonal set to $-\infty$, then softmax; (b) softmax of all scores, weights above the diagonal set to zero, rows divided by their new sums. Show numerically that they agree and prove it in one line. What does (b) without the division compute, and why is that still a leak?

### 6.2 Several heads

- Implement `MultiHeadAttention(d_in, d_out, context_length, dropout, num_heads, qkv_bias=False)` with the attribute names of Materials, Appendix A (List 5 loads GPT-2's weights into it by name): one `nn.Linear` each for queries, keys and values, the mask as a buffer, all heads computed at once with `view` and `transpose`, dropout on the attention weights, an output projection. Run the `MultiHeadAttention` part of the contract check (Appendix A).
- Count its parameters, check the count with the module and evaluate it for GPT-2 small ($d_\text{in} = d_\text{out} = 768$, 12 heads, `qkv_bias=True`). Why does the count not depend on the number of heads?
- Train your model with `MultiHeadAttention(64, 64, 64, 0.0, h)` for $h = 1$ and $h = 4$: same parameters, different loss. Our run: 2.40 and 2.20. Plot the four heads' attention matrices for the same window as in Task 5.3. Do the heads do different things? Describe what you see.

### 6.3 The cost of attention

- With `MultiHeadAttention(256, 256, 2048, 0.0, 4)`, batch size 1, inside `torch.no_grad()`: time the forward pass for $T \in \{64, 128, 256, 512, 1024, 2048\}$ (median of a few repeats after a warm-up) and plot the time against $T$ on log–log axes. Explain the slope at the largest $T$ with the costs of the projections ($T d^2$) and of the scores ($T^2 d$). Then compute the memory of the score tensor ($B \cdot h \cdot T^2$ `float32` values) at $T = 2048$ and for one GPT-2 small layer at $T = 1024$. Which of the two, time or memory, is the practical limit on context length?

- **Checkpoint:** the `MultiHeadAttention` part of the contract check (Materials, Appendix A) passes; four copies of your one-head layer from Task 5.3, with their weights copied into `MultiHeadAttention`, give the same output as the module; `leak_test` on a model with your `MultiHeadAttention` is not below $\ln V$; and the gradient of the output at position $t$ with respect to the inputs after $t$ is exactly zero (Materials, Appendix B).

## Lab 7: Going deep

### 7.1 Communication, then computation

A Transformer block is attention (positions exchange information) followed by a position-wise feed-forward network (each position computes on its own): `nn.Linear(d, 4d)`, a nonlinearity (`nn.GELU` for now; your own in Task 7.3), `nn.Linear(4d, d)`.

- Stack $L = 1$ and $L = 4$ blocks with 4 heads and **no** residual connections, and train both. Our run: 2.02 and 3.36. What is 3.36? Compare it with the entropy of the character frequencies of the text. For $L = 4$, plot the gradient norm of each block's first feed-forward weight matrix at initialisation, on a logarithmic axis, for the loss of one batch.
- Add residual connections, $x \leftarrow x + \mathrm{Attn}(x)$ and $x \leftarrow x + \mathrm{FF}(x)$, and repeat both runs and the gradient plot. Our run: 1.98 and 1.85. Explain the difference with the Jacobian of one block, $\partial F / \partial x$ without and $I + \partial F / \partial x$ with the shortcut.

### 7.2 LayerNorm

- Implement `LayerNorm(emb_dim)` as in Materials, Appendix A (biased variance, $\varepsilon = 10^{-5}$, `scale` and `shift`), and check it against `nn.LayerNorm` within $10^{-6}$ for random scale and shift.
- Put it in front of each sub-layer (Pre-LN: $x \leftarrow x + \mathrm{Attn}(\mathrm{LN}(x))$, $x \leftarrow x + \mathrm{FF}(\mathrm{LN}(x))$, and a final LayerNorm before the output layer). Train 4 blocks with and without LayerNorm at the learning rates $3 \cdot 10^{-3}$ and $3 \cdot 10^{-2}$. Our run at $3 \cdot 10^{-3}$: 1.85 with and 1.85 without. Which run diverges or stalls at $3 \cdot 10^{-2}$? What does LayerNorm buy, if not a lower loss at the default learning rate?

### 7.3 GELU and the feed-forward block

- Implement `GELU` (the tanh approximation used by GPT-2) and `FeedForward(cfg)` as in Materials, Appendix A, and run the contract check. Derive the derivative of the exact GELU $x\,\Phi(x)$ by hand and compare it with autograd on your `GELU` (the tanh approximation is within $5 \cdot 10^{-4}$ of the exact function). Check that `FeedForward` has $8d^2 + 5d$ parameters and compare it with the attention layer of Task 6.2 at $d = 768$: where are the parameters of a block?
- Train your 4-block model once with GELU and once with ReLU in the feed-forward network. Report both losses and say honestly what you see; at this scale the two are usually within noise. What does GELU do differently for negative inputs?

### 7.4 Put it together

- Collect all your runs in one table (model, parameters, train loss, validation loss, seconds) in the order you built them. Which single addition bought the most per parameter, and which per second of training?
- Train your best configuration for a longer budget (default: 3000 steps, several minutes) and generate 500 characters from it. Report train and validation loss and comment on the text: what has the model learned, and what has it not?

### 7.5 Other normalisations *(optional)*

- Repeat the learning-rate experiment of Task 7.2 with Post-LN blocks, $x \leftarrow \mathrm{LN}(x + F(x))$, and relate what you see to Xiong et al. (2020).
- Show in a few lines why BatchNorm is the wrong choice for sequences: normalise a batch of shape $(B, T, d)$ per feature over batch and time, add zero-padded sequences to the batch and show that the outputs of the unpadded sequences change, while a LayerNorm output never depends on the other tokens or sequences.

### 7.6 Copying needs two layers *(optional)*

- A synthetic task: each training sequence is a block of $n$ uniformly random tokens followed by the same block again, with $n$ drawn anew for every batch from $16, \dots, 32$. Train a 1-block and a 2-block model (your Pre-LN blocks, default settings, 1500 steps) on this task and plot the loss per position for a test batch. Nothing can be predicted in the first half; a model that has learned to copy gets a loss near 0 on the second half. Which model can do it, and why does copying from a previous occurrence need two layers of attention? Our run, second half: about 3.0 with one block, 0.2 with two. (Why does the task become trivial for one block when $n$ is fixed?)

### 7.7 Your own experiment *(optional)*

About one hour of work: one question this list raised, answered with one plot or table and a paragraph. Not a hyperparameter sweep; change something about the mechanism (Materials, Appendix C has ideas).

- **Checkpoint:** the contract check of Materials, Appendix A, passes for `MultiHeadAttention`, `LayerNorm`, `GELU` and `FeedForward`; `leak_test` on your final model is not below $\ln V$; and the model generates text.